In [1]:
import numpy as nmp
from netCDF4 import Dataset as ncd
from netCDF4 import num2date, date2num
import matplotlib.pyplot as plt
import os


from specialized_input_data.specialized_input_data import *

### Mineral monochromatic Look-Up Table (Mie)

In [2]:
sufix = "empirical_moreWL"
inp = ncd('Mie_IrOx_LUT/mineral_monochromatic_table_empirical_moreWL.nc', 'r') 
#This is a LUT of Mie properties for different: IrOx mass%, size bins and spectral bands
dens = (inp.variables['DENS'][:]).filled() # [S] (total dust, constant for all bins) (g cm-3)
reff = (inp.variables['REFF'][:]).filled() # [S] (um)
qext = (inp.variables['QEXT'][:,:]).filled() # [S,B_mono]
qsca = (inp.variables['QSCA'][:,:]).filled()
qasy = (inp.variables['QASY'][:,:]).filled()
vext = (inp.variables['VEXT'][:,:,:]).filled() #[S,B_mono,Im]
vsca = (inp.variables['VSCA'][:,:,:]).filled() 
vasy = (inp.variables['VASY'][:,:,:]).filled()
#Only if they were created in the Mie LUT (sometimes I don't fore time issues):
"""wext = (inp.variables['WEXT'][:,:,:,:]).filled() # [S,B_mono,Im,Im]
wsca = (inp.variables['WSCA'][:,:,:,:]).filled()
wasy = (inp.variables['WASY'][:,:,:,:]).filled()"""

inp.close()

#Depending on which approach you use, you have to change the aux function ()

### Input of mass concentrations

In [3]:
#Here I should take only the mass loads from the selected pixel from monarch
#That will provide: mass% of hema, goet and host in the column. With mdnen I calculate vol%
# for each of monarch 8 size bins. They will be daily means (so no T=12)
#The only dims of the Thessaloniki mass% are Day (the day) and S (size bin)(8)

In [4]:
inp = ncd(r'C:/Users/Usuario/Desktop/BSC_secondment/monarch_massfract_Thess_dod01.nc', 'r') 
#Dims are: 19 datetime64 (Day) x 8 size bins (S)
T = inp.dimensions['Day'].size #dates
S = inp.dimensions['S'].size #monarch size bins
xhema = (inp.variables['hema'][:]).filled()
xgoet = (inp.variables['goet'][:]).filled()
xhost = (inp.variables['host'][:]).filled()
xiox = xhema + xgoet ###
mdust = (inp.variables['dust_mass'][:]).filled()*0.1 #transforming kg/m2 to g/cm2

day_var = inp.variables['Day']
day_units = day_var.units
day_calendar = getattr(day_var, 'calendar', 'standard')
day_dates = num2date(day_var[:], units=day_units, calendar=day_calendar)
inp.close()

### MASS DENSITIES ###
#Only the mineral species given in EMIT+MONARC (11 witout mixes)

                 # Illi   Kaol   Mont   Chlo   Verm   Calc   Quar   Feld   Gyps   Hema   Goet
mden = nmp.array([2.570, 2.630, 2.570, 2.420, 2.300, 2.710, 2.670, 2.680, 2.308, 5.250, 4.180]) # (g cm-3)

In [ ]:
"""
xgoet[:]=0.005
xhema[:]=0.02
"""

'\nxgoet[:]=0.005\nxhema[:]=0.02\n\nstored = xhema.copy()\nxhema = xgoet.copy()\nxgoet = stored.copy()\n'

In [6]:
"""
from matplotlib.lines import Line2D

base_path = r'C:/Users/Usuario/Desktop/BSC_secondment/'
files = {
    'Thess': 'monarch_massfract_thess_aeronet.nc',
    'DOD 01': 'monarch_massfract_thess_dod01.nc',
    'DOD 03': 'monarch_massfract_thess_dod03.nc',
    'DOD 05': 'monarch_massfract_thess_dod05.nc'
}

green = ['#8FBC8F', '#4F8B5B', '#276738', '#154421']
red = ['#E57373', '#D32F2F', '#A31515', '#720D0D']
gray = ['#202020', '#808080', '#505050', '#202020']

bins = nmp.arange(1, 9)
fig, ax1 = plt.subplots(figsize=(10, 6))
ax2 = ax1.twinx()

for i, (name, filename) in enumerate(files.items()):
    inp = ncd(os.path.join(base_path, filename), 'r')

    xhema = inp.variables['hema'][:].filled() * 100
    xgoet = inp.variables['goet'][:].filled() * 100
    mdust = inp.variables['dust_mass'][:].filled() * 0.1

    mean_xhema, std_xhema = nmp.mean(xhema, axis=0), nmp.std(xhema, axis=0)
    mean_xgoet, std_xgoet = nmp.mean(xgoet, axis=0), nmp.std(xgoet, axis=0)
    mean_mdust, std_mdust = nmp.mean(mdust, axis=0), nmp.std(mdust, axis=0)

    color_hema = 'blue' if i == 0 else green[i]
    color_goet = 'orange' if i == 0 else red[i]
    marker = 's' if i == 0 else 'o'
    linestyle = '--' if i == 0 else '-'

    ax1.errorbar(bins, mean_xhema, yerr=std_xhema, fmt=marker + linestyle, color=color_hema, capsize=4, linewidth=2)
    ax1.errorbar(bins, mean_xgoet, yerr=std_xgoet, fmt=marker + linestyle, color=color_goet, capsize=4, linewidth=2)
    ax2.errorbar(bins, mean_mdust, yerr=std_mdust, fmt=marker + linestyle, color=gray[i], linestyle=linestyle, capsize=4, linewidth=2)

    inp.close()

ax1.set_xlabel("Bin")
ax1.set_ylabel("Mass fraction (%)")
ax2.set_ylabel("Total dust mass (g/cm²)")
ax1.set_xticks(bins)
ax1.grid(alpha=0.3)

# Legend 1: variables and units
legend_left = [
    Line2D([0], [0], color='#D32F2F', linewidth=2, label='Goethite (%)'),
    Line2D([0], [0], color='#276738', linewidth=2, label='Hematite (%)'),
    Line2D([0], [0], color='black', linewidth=2, label='Total dust (g/cm²)')
]

leg1 = ax1.legend(handles=legend_left, loc='center left', bbox_to_anchor=(0.01, 0.5))
ax1.add_artist(leg1)

# Legend 2: experiment markers
legend_right = [
    Line2D([0], [0], color='black', marker='s', linestyle='--', markersize=7, label='AERONET filtered'),
    Line2D([0], [0], color='black', marker='o', linestyle='-', markersize=7, label='MONARCH DOD filtered')
]

ax2.legend(handles=legend_right, loc='center left', bbox_to_anchor=(0.01, 0.25))

plt.tight_layout()
plt.show()"""


'\nfrom matplotlib.lines import Line2D\n\nbase_path = r\'C:/Users/Usuario/Desktop/BSC_secondment/\'\nfiles = {\n    \'Thess\': \'monarch_massfract_thess_aeronet.nc\',\n    \'DOD 01\': \'monarch_massfract_thess_dod01.nc\',\n    \'DOD 03\': \'monarch_massfract_thess_dod03.nc\',\n    \'DOD 05\': \'monarch_massfract_thess_dod05.nc\'\n}\n\ngreen = [\'#8FBC8F\', \'#4F8B5B\', \'#276738\', \'#154421\']\nred = [\'#E57373\', \'#D32F2F\', \'#A31515\', \'#720D0D\']\ngray = [\'#202020\', \'#808080\', \'#505050\', \'#202020\']\n\nbins = nmp.arange(1, 9)\nfig, ax1 = plt.subplots(figsize=(10, 6))\nax2 = ax1.twinx()\n\nfor i, (name, filename) in enumerate(files.items()):\n    inp = ncd(os.path.join(base_path, filename), \'r\')\n\n    xhema = inp.variables[\'hema\'][:].filled() * 100\n    xgoet = inp.variables[\'goet\'][:].filled() * 100\n    mdust = inp.variables[\'dust_mass\'][:].filled() * 0.1\n\n    mean_xhema, std_xhema = nmp.mean(xhema, axis=0), nmp.std(xhema, axis=0)\n    mean_xgoet, std_xgoet = 

In [7]:
"""inp = ncd(r'C:/Users/Usuario/Desktop/BSC_secondment/monarch_massfract_Thess_dod01.nc', 'r') 
#Dims are: 19 datetime64 (Day) x 8 size bins (S)
T = inp.dimensions['Day'].size #dates
S = inp.dimensions['S'].size #monarch size bins
xhema = (inp.variables['hema'][:]).filled()
xgoet = (inp.variables['goet'][:]).filled()
xhost = (inp.variables['host'][:]).filled()
dod = (inp.variables['dod_550nm'][:]).filled()

bin_index = 4

for y, label in [(xhema[:, bin_index], "hematite"), (xgoet[:, bin_index], "goethite")]:
    plt.scatter(dod, y, label=label)
    m, b = nmp.polyfit(dod, y, 1)
    plt.plot(dod, m*nmp.array(dod) + b)

plt.xlabel("DOD 550nm")
plt.ylabel("mass fraction")
plt.legend()
plt.show()"""

'inp = ncd(r\'C:/Users/Usuario/Desktop/BSC_secondment/monarch_massfract_Thess_dod01.nc\', \'r\') \n#Dims are: 19 datetime64 (Day) x 8 size bins (S)\nT = inp.dimensions[\'Day\'].size #dates\nS = inp.dimensions[\'S\'].size #monarch size bins\nxhema = (inp.variables[\'hema\'][:]).filled()\nxgoet = (inp.variables[\'goet\'][:]).filled()\nxhost = (inp.variables[\'host\'][:]).filled()\ndod = (inp.variables[\'dod_550nm\'][:]).filled()\n\nbin_index = 4\n\nfor y, label in [(xhema[:, bin_index], "hematite"), (xgoet[:, bin_index], "goethite")]:\n    plt.scatter(dod, y, label=label)\n    m, b = nmp.polyfit(dod, y, 1)\n    plt.plot(dod, m*nmp.array(dod) + b)\n\nplt.xlabel("DOD 550nm")\nplt.ylabel("mass fraction")\nplt.legend()\nplt.show()'

### Global optical interpolation (functions)

In [8]:
def GlbOpts(XHM,XGT,s1,wvl): #mass% of Hema and Goet, idx of a size bin, idx of wvl
 """
 Interpolates from the LUT the Mie properties to the given mass%s of H and G, size and monochromatic wavelength.
 The Mie properties are:
 - EXT: extinction efficiency
 - SCA: scattering "
 - ASY: asymmetry parameter
 """

 #Keeps the mass fractions inside the LUT range (0%->10%), without extrapolation
 XHM = float(nmp.clip(XHM, igrd[0], igrd[-1]))
 XGT = float(nmp.clip(XGT, igrd[0], igrd[-1]))

 #Finds the lower grid indices for bilinear interpolation
 h = int(nmp.clip(nmp.searchsorted(igrd, XHM, side='right')-1, 0, Im-2))
 g = int(nmp.clip(nmp.searchsorted(igrd, XGT, side='right')-1, 0, Im-2))
 #Takes the two bins from the grid that enclose the value
 HG1 = igrd[h]
 HG2 = igrd[h+1]
 GG1 = igrd[g]
 GG2 = igrd[g+1]

 #Calculates the weights of the 4 surrounding LUT points
 DHG = (HG2-HG1)*(GG2-GG1)
 W11 = (HG2-XHM)*(GG2-XGT)/DHG
 W12 = (HG2-XHM)*(XGT-GG1)/DHG
 W21 = (XHM-HG1)*(GG2-XGT)/DHG
 W22 = (XHM-HG1)*(XGT-GG1)/DHG

 #Interpolates extinction, scattering and asymmetry for the selected size and wavelength
 EXT = wext[s1,wvl,h,g]*W11 + wext[s1,wvl,h,g+1]*W12 + wext[s1,wvl,h+1,g]*W21 + wext[s1,wvl,h+1,g+1]*W22
 SCA = wsca[s1,wvl,h,g]*W11 + wsca[s1,wvl,h,g+1]*W12 + wsca[s1,wvl,h+1,g]*W21 + wsca[s1,wvl,h+1,g+1]*W22
 ASY = wasy[s1,wvl,h,g]*W11 + wasy[s1,wvl,h,g+1]*W12 + wasy[s1,wvl,h+1,g]*W21 + wasy[s1,wvl,h+1,g+1]*W22


 return EXT, SCA, ASY 

In [9]:
def GlbOpts_IrOx(XHM,XGT,s1,wvl): #mass% of Hema and Goet, idx of a size bin, idx of wvl
 #Does the exact same of previous function but for a single mass of IrOx species added up

 #Exactly all same steps than GlobOpts (see there) but for the sum of all iron oxides:
 XIOX = float(nmp.clip(XHM + XGT, igrd[0], igrd[-1])) 

 iox = int(nmp.clip(nmp.searchsorted(igrd, XIOX, side='right')-1, 0, Im-2))
 IOX1 = igrd[iox]
 IOX2 = igrd[iox+1]

 DIOX = IOX2-IOX1
 W1 = (IOX2-XIOX)/DIOX
 W2 = (XIOX-IOX1)/DIOX

 EXT_IOX = vext[s1,wvl,iox]*W1 + vext[s1,wvl,iox+1]*W2
 SCA_IOX = vsca[s1,wvl,iox]*W1 + vsca[s1,wvl,iox+1]*W2
 ASY_IOX = vasy[s1,wvl,iox]*W1 + vasy[s1,wvl,iox+1]*W2

 return EXT_IOX, SCA_IOX, ASY_IOX 

### Global optical properties

In [10]:
# Allocate results for all days
names = ['homogenous_dust', 'mixed_dust']
#to cencel/include homogenous case, just comment/descomment 'continue' in the 2 c loops

gext_all = nmp.zeros([2, T, B_mono])
gsca_all = nmp.zeros([2, T, B_mono])
gasy_all = nmp.zeros([2, T, B_mono])


# This loop is adapted for MONARCH input where the pixel is already selected
# and dust masses have been integrated over the vertical levels.
# The bins are adapted to DustCOMM bins and the mass fractions
# are calculated for hematite, goethite and host.

for t in nmp.arange(T):

    print('Processing day: ', day_dates[t])

    # Optical contributions for each size bin and wavelength
    dext, dsca, dasy = (nmp.zeros([S, B_mono]), nmp.zeros([S, B_mono]), nmp.zeros([S, B_mono]))
    mext, msca, masy = (nmp.zeros([S, B_mono]), nmp.zeros([S, B_mono]), nmp.zeros([S, B_mono]))

    for s in nmp.arange(S):

        DEN = dens[s]          # g cm-3
        REF = reff[s] * 1e-4  # um -> cm
        QTK = 3.0 / (4.0 * DEN * REF)

        MDU = mdust[t, s]     # g cm-2

        XHM = xhema[t, s]
        XGT = xgoet[t, s]

        for b in nmp.arange(B_mono):

            # Homogeneous mixture
            EXT = qext[s, b]
            SCA = qsca[s, b]
            ASY = qasy[s, b]

            dext[s, b] = EXT * QTK * MDU
            dsca[s, b] = SCA * QTK * MDU
            dasy[s, b] = ASY

            # Varying mineral mixture
            #EXT, SCA, ASY = GlbOpts(XHM, XGT, s, b)
            EXT, SCA, ASY = GlbOpts_IrOx(XHM, XGT, s, b)
            #Dont need to change anything from V to W (INB=1 -> 2) if the function is already changed 

            mext[s, b] = EXT * QTK * MDU
            msca[s, b] = SCA * QTK * MDU
            masy[s, b] = ASY

    # Reference extinction: all size bins, mixed composition
    gext = nmp.sum(mext, axis=0)

    # --------------------------------------------------
    # Calculate the homogenous and mixed cases

    for c in nmp.arange(2):

        if c == 0:
            #continue
            # Homogeneous
            bext = nmp.sum(dext, axis=0)
            bsca = nmp.sum(dsca, axis=0)
            gsca = nmp.divide(
                bsca * gext, bext,
                out=nmp.zeros_like(bsca),
                where=bext != 0
            )
            scat = dsca
            asym = dasy

        if c == 1:
            # Mixed minerals or sum of irox species as one (iox)
            bext = nmp.sum(mext, axis=0)
            bsca = nmp.sum(msca, axis=0)
            gsca = nmp.divide(
                bsca * gext, bext,
                out=nmp.zeros_like(bsca),
                where=bext != 0
            )
            scat = msca
            asym = masy

        # Scattering-weighted asymmetry parameter
        scat_sum = nmp.sum(scat, axis=0)
        gasy = nmp.divide(
            nmp.sum(asym * scat, axis=0),
            scat_sum,
            out=nmp.zeros(B_mono),
            where=scat_sum != 0
        )

        # Store results for this day and PM case
        gext_all[c, t, :] = gext
        gsca_all[c, t, :] = gsca
        gasy_all[c, t, :] = gasy


# --------------------------------------------------
# Write one NetCDF file per PM case, containing all days

for c in nmp.arange(2):

    #if c == 0:
    #    continue
    
    out = ncd(f'results_IrOx/%s_{sufix}.nc' % names[c], 'w', format='NETCDF4')

    out.createDimension('Day', T)
    out.createDimension('B_mono', B_mono)

    # Date coordinate
    DAY = out.createVariable('Day', 'f8', ('Day',))
    DAY.units = day_units
    DAY.calendar = day_calendar
    DAY.long_name = 'Date'
    DAY[:] = date2num(day_dates, units=day_units, calendar=day_calendar)

    # Wavelength coordinate
    WVL = out.createVariable('wavelength', 'f4', ('B_mono',))
    WVL.units = 'um'
    WVL.long_name = 'Monochromatic wavelength'
    WVL[:] = wvl_micron

    # Extinction optical depth
    GEXT = out.createVariable('GEXT', 'f4', ('Day', 'B_mono'))
    GEXT.units = '-'
    GEXT.long_name = 'Dust column extinction optical depth'

    # Scattering optical depth
    GSCA = out.createVariable('GSCA', 'f4', ('Day', 'B_mono'))
    GSCA.units = '-'
    GSCA.long_name = 'Dust column scattering optical depth'

    # Asymmetry parameter
    GASY = out.createVariable('GASY', 'f4', ('Day', 'B_mono'))
    GASY.units = '-'
    GASY.long_name = 'Dust asymmetry parameter'

    GEXT[:, :] = gext_all[c, :, :]
    GSCA[:, :] = gsca_all[c, :, :]
    GASY[:, :] = gasy_all[c, :, :]

    out.close()

Processing day:  2007-03-10 00:00:00
Processing day:  2007-03-29 00:00:00
Processing day:  2007-03-30 00:00:00
Processing day:  2007-04-01 00:00:00
Processing day:  2007-04-04 00:00:00
Processing day:  2007-05-04 00:00:00
Processing day:  2007-05-05 00:00:00
Processing day:  2007-05-06 00:00:00
Processing day:  2007-05-07 00:00:00
Processing day:  2007-05-20 00:00:00
Processing day:  2007-05-28 00:00:00
Processing day:  2007-06-05 00:00:00
Processing day:  2007-06-19 00:00:00
Processing day:  2007-06-20 00:00:00
Processing day:  2007-06-21 00:00:00
Processing day:  2007-06-22 00:00:00
Processing day:  2007-06-23 00:00:00
Processing day:  2007-06-24 00:00:00
Processing day:  2007-06-25 00:00:00
Processing day:  2007-06-26 00:00:00
Processing day:  2007-06-27 00:00:00
Processing day:  2007-07-23 00:00:00
Processing day:  2007-07-24 00:00:00
Processing day:  2007-07-25 00:00:00
Processing day:  2007-08-22 00:00:00
Processing day:  2007-08-23 00:00:00
Processing day:  2007-08-24 00:00:00
P